# Dataset Audit and Leakage-Free Splitting

Audits the local NEU-DET dataset copy, verifies licence-gate metadata in `configs/dataset.yaml`, detects exact and near-duplicate images, and creates reproducible train/val/golden-test manifests plus the natural hard subset and controlled corruption suite.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path('/kaggle/working/EdgeVisionInspector'), Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists()), Path('/kaggle/working/EdgeVisionInspector'))
os.environ['EDGEVISION_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from edgevision.config import load_yaml
from edgevision.data.annotations import parse_dataset
from edgevision.data.audit import analyze_duplicates, run_data_audit
from edgevision.data.splits import save_split_manifests, stratified_group_split

dataset_cfg = load_yaml(REPO_ROOT / 'configs/dataset.yaml')
RAW_DIR = Path('/kaggle/input/neu-det') if Path('/kaggle/input/neu-det').exists() else (REPO_ROOT / dataset_cfg['paths']['raw_dir'])
print('Using raw dataset directory:', RAW_DIR)
print('Licence gate status:', dataset_cfg['licence_gate'])

In [ ]:
parse_result = parse_dataset(RAW_DIR, dataset_cfg)
audit_summary = run_data_audit(
    parse_result,
    dataset_cfg=dataset_cfg,
    report_path=REPO_ROOT / 'reports/data_audit.md',
    figures_dir=REPO_ROOT / 'reports/figures',
)
print('Valid images:', audit_summary['num_valid_images'])
print('Total boxes:', audit_summary['total_instances'])
print('Class distribution:', audit_summary['class_distribution'])

In [ ]:
split_cfg = dataset_cfg['split']
analyze_duplicates(
    parse_result.records,
    phash_max_dist=int(split_cfg['near_duplicate_phash_max_dist']),
    cosine_min=float(split_cfg['near_duplicate_cosine_min']),
)
split_res = stratified_group_split(
    records=parse_result.records,
    num_classes=len(dataset_cfg['classes']),
    train_ratio=float(split_cfg['train_ratio']),
    val_ratio=float(split_cfg['val_ratio']),
    test_ratio=float(split_cfg['test_ratio']),
    seed=int(split_cfg['seed']),
)
paths = save_split_manifests(split_res, manifests_dir=REPO_ROOT / 'data/manifests', dataset_cfg=dataset_cfg)
print('Saved manifests:', {k: str(v) for k, v in paths.items()})